# Tải riêng EraX từ Hugging Face để chạy offline
Internet **On**, không cần GPU. Output là thư mục `EraX-Translator-V1.0/` chứa trọng số, tokenizer, processor và metadata; không chứa thư viện Python. Add Output này cùng wheel bundle vào notebook L4 offline. Tuân thủ giấy phép Gemma khi chia sẻ model.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub>=1.32,<3'], check=True)
import os
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN').strip()
    if token:
        os.environ['HF_TOKEN'] = token
    del token
except Exception:
    print('Dùng HF_TOKEN môi trường nếu có, không in token.')


In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
import json, hashlib
MODEL_ID = 'erax-ai/EraX-Translator-V1.0'
REVISION = 'main'
base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
destination = base / 'EraX-Translator-V1.0'
info = HfApi().model_info(MODEL_ID, revision=REVISION)
print('Model revision:', info.sha)
snapshot_download(MODEL_ID, revision=info.sha, local_dir=str(destination),
                  ignore_patterns=['*.gguf', '*.pth', '*.onnx', 'original/*'])
assert (destination / 'config.json').exists()
assert list(destination.glob('*.safetensors'))
index = destination / 'model.safetensors.index.json'
if index.exists():
    for shard in set(json.loads(index.read_text())['weight_map'].values()):
        assert (destination / shard).exists(), f'Thiếu shard: {shard}'
files = []
for path in sorted(destination.rglob('*')):
    if not path.is_file() or '.cache' in path.parts or path.name == 'erax-model-manifest.json':
        continue
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    files.append(dict(file=str(path.relative_to(destination)), bytes=path.stat().st_size, sha256=digest.hexdigest()))
manifest = dict(model_id=MODEL_ID, revision=info.sha, files=files)
(destination / 'erax-model-manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Output:', destination)
print('Dung lượng GiB:', round(sum(f['bytes'] for f in files) / 1024**3, 2))
print('Save Version để lưu Output; Add Input vào notebook 2_dich_han_viet.ipynb.')
